In [1]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import plotly.graph_objects as go

# Ensure the saved_models directory exists
os.makedirs('saved_models', exist_ok=True)

# Check hardware availability
device = torch.device("cuda")
print(f"System ready. Using device: {device}")

System ready. Using device: cuda


In [2]:
class RockfallDataset(Dataset):
    def __init__(self, root_dir='./Dataset', num_points=4096):
        self.num_points = num_points
        self.files = []
        self.labels = []
        
        # Class mapping based on directory structure
        classes = {'class_0_safe_unity': 0, 'class_1_hazard_unity': 1}
        
        for cls_name, cls_label in classes.items():
            cls_dir = os.path.join(root_dir, cls_name)
            for file_path in glob.glob(os.path.join(cls_dir, '*.npy')):
                self.files.append(file_path)
                self.labels.append(cls_label)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        point_cloud = np.load(self.files[idx]) 
        
        # Standardize point count to exactly 4096
        if len(point_cloud) > self.num_points:
            indices = np.random.choice(len(point_cloud), self.num_points, replace=False)
            point_cloud = point_cloud[indices]
        elif len(point_cloud) < self.num_points:
            indices = np.random.choice(len(point_cloud), self.num_points, replace=True)
            point_cloud = point_cloud[indices]
            
        # Normalize to unit sphere for PointNet input
        centroid = np.mean(point_cloud, axis=0)
        point_cloud = point_cloud - centroid
        m = np.max(np.sqrt(np.sum(point_cloud**2, axis=1)))
        point_cloud = point_cloud / m
        
        # Format for PyTorch: (Channels, Points) -> (3, 4096)
        point_cloud = torch.tensor(point_cloud.T, dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        
        return point_cloud, label

print("Data pipeline initialized.")

Data pipeline initialized.


In [3]:
class PointNetVanilla(nn.Module):
    def __init__(self, num_classes=2):
        super(PointNetVanilla, self).__init__()
        
        self.conv1 = nn.Conv1d(3, 64, 1)
        self.conv2 = nn.Conv1d(64, 64, 1)
        self.conv3 = nn.Conv1d(64, 64, 1)
        self.conv4 = nn.Conv1d(64, 128, 1)
        self.conv5 = nn.Conv1d(128, 1024, 1)
        
        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(64)
        self.bn3 = nn.BatchNorm1d(64)
        self.bn4 = nn.BatchNorm1d(128)
        self.bn5 = nn.BatchNorm1d(1024)
        
        self.fc1 = nn.Linear(1024, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, num_classes)
        
        self.dropout = nn.Dropout(p=0.3)
        self.bn6 = nn.BatchNorm1d(512)
        self.bn7 = nn.BatchNorm1d(256)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = F.relu(self.bn5(self.conv5(x)))
        
        # Global Max Pooling for permutation invariance
        x = torch.max(x, 2, keepdim=True)[0]
        x = x.view(-1, 1024) 
        
        x = F.relu(self.bn6(self.fc1(x)))
        x = self.dropout(x)
        x = F.relu(self.bn7(self.fc2(x)))
        x = self.dropout(x)
        x = self.fc3(x)
        
        return F.log_softmax(x, dim=1)

print("PointNet architecture initialized.")

PointNet architecture initialized.


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import os

# --- NEW IMPORTS FOR CONFUSION MATRIX ---
import numpy as np
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 16 
epochs = 20
learning_rate = 0.00001

# Initialize Dataset
# (Assuming RockfallDataset and PointNetVanilla are defined above in your script)
full_dataset = RockfallDataset(root_dir='./Dataset')

if len(full_dataset) == 0:
    print("ERROR: No .npy files found in the Dataset folders.")
else:
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = PointNetVanilla(num_classes=2).to(device)
    criterion = nn.NLLLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    print(f"Starting training on {len(full_dataset)} point clouds using {device}...")

    # Lists to hold the final epoch's predictions for the confusion matrix
    final_val_preds = []
    final_val_labels = []

    for epoch in range(epochs):
        model.train()
        train_loss, correct, total = 0.0, 0, 0
        
        for points, labels in train_loader:
            points, labels = points.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(points)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
        train_acc = 100 * correct / total
        
        # Validation Pass
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        
        # Clear the lists for the current epoch
        epoch_val_preds = []
        epoch_val_labels = []
        
        with torch.no_grad():
            for points, labels in val_loader:
                points, labels = points.to(device), labels.to(device)
                outputs = model(points)
                
                # Calculate validation loss
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
                
                # Store predictions and true labels for the confusion matrix
                epoch_val_preds.extend(predicted.cpu().numpy())
                epoch_val_labels.extend(labels.cpu().numpy())
                
        val_acc = 100 * val_correct / val_total if val_total > 0 else 0
        
        # Calculate average losses for printing
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader) if len(val_loader) > 0 else 0
        
        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.1f}% | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.1f}%")
        
        # If this is the last epoch, save the predictions for plotting
        if epoch == epochs - 1:
            final_val_preds = epoch_val_preds
            final_val_labels = epoch_val_labels

    # --- CONFUSION MATRIX GENERATION ---
    print("\nGenerating Confusion Matrix for the final Validation Set...")
    cm = confusion_matrix(final_val_labels, final_val_preds)
    
    plt.figure(figsize=(8, 6))
    # You can update the tick labels if you have specific class names (e.g., ['Non-Rockfall', 'Rockfall'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Class 0', 'Class 1'], 
                yticklabels=['Class 0', 'Class 1'])
    plt.xlabel('Predicted Labels')
    plt.ylabel('True Labels')
    plt.title('Validation Confusion Matrix (Final Epoch)')
    
    cm_save_path = './saved_models/confusion_matrix.png'
    os.makedirs(os.path.dirname(cm_save_path), exist_ok=True)
    plt.savefig(cm_save_path, bbox_inches='tight')
    plt.close() # Close to free up memory
    print(f"Confusion Matrix saved to: {cm_save_path}")

    # --- TORCHSCRIPT EXPORT LOGIC ---
    print("\nCompiling model to TorchScript for Live Inference...")
    
    # 1. Put the model in evaluation mode (critical for batch norm/dropout layers)
    model.eval()
    
    # 2. Create a dummy tensor that perfectly matches your 4,096 Unity points (Batch=1, Channels=3, Points=4096)
    dummy_input = torch.randn(1, 3, 4096, dtype=torch.float32).to(device)
    
    # 3. Trace the model (Bakes the architecture and weights together)
    with torch.no_grad():
        traced_model = torch.jit.trace(model, dummy_input)
    
    # 4. Save the compiled standalone .pt file
    standalone_save_path = './saved_models/rockfall_pointnet_unity.pt'
    
    traced_model.save(standalone_save_path)
    
    print("="*60)
    print(f"Training & Compilation Complete!")
    print(f"Standalone Model Saved to: {standalone_save_path}")
    print("="*60)

Starting training on 2000 point clouds using cuda...
Epoch [1/20] | Train Loss: 0.7036 | Train Acc: 54.2% | Val Loss: 0.6704 | Val Acc: 56.2%
Epoch [2/20] | Train Loss: 0.6294 | Train Acc: 64.7% | Val Loss: 0.6273 | Val Acc: 68.0%
Epoch [3/20] | Train Loss: 0.5549 | Train Acc: 73.1% | Val Loss: 0.5810 | Val Acc: 76.2%
Epoch [4/20] | Train Loss: 0.4737 | Train Acc: 80.8% | Val Loss: 0.5363 | Val Acc: 79.0%
Epoch [5/20] | Train Loss: 0.4429 | Train Acc: 82.2% | Val Loss: 0.4817 | Val Acc: 86.0%
Epoch [6/20] | Train Loss: 0.4024 | Train Acc: 84.4% | Val Loss: 0.4786 | Val Acc: 82.8%
Epoch [7/20] | Train Loss: 0.3719 | Train Acc: 86.7% | Val Loss: 0.4291 | Val Acc: 88.0%
Epoch [8/20] | Train Loss: 0.3477 | Train Acc: 87.5% | Val Loss: 0.3964 | Val Acc: 89.5%
Epoch [9/20] | Train Loss: 0.3281 | Train Acc: 89.2% | Val Loss: 0.3636 | Val Acc: 89.5%
Epoch [10/20] | Train Loss: 0.3048 | Train Acc: 89.4% | Val Loss: 0.3624 | Val Acc: 90.2%
Epoch [11/20] | Train Loss: 0.3047 | Train Acc: 89.9% | 